# map of wroclaw, points of laboratories and hospital

* pkty, szpital i droga

In [1]:
from pathlib import Path
import pandas as pd
import folium
import osmnx as ox
import numpy as np


# ============================================================
# 1. USTAWIENIA
# ============================================================

OUT_DIR = Path("wroclaw_map_output")
OUT_DIR.mkdir(exist_ok=True)



addresses = [
    "pl. Grunwaldzki 18-20, Wroclaw, Poland",
    "Plac Hirszfelda 16/17, Wroclaw, Poland",
    "Św. Macieja 8, Wroclaw, Poland",
    "Bierutowska 17, Wroclaw, Poland",
    "Bonczyka 20, Wroclaw, Poland",
    "Budziszyńska 62a, Wroclaw, Poland",
    "Buforowa 75, Wroclaw, Poland",
    "Canaletta 4, Wroclaw, Poland",
    "Chorwacka 41c, Wroclaw, Poland",
    "Dobrzyńska 21/23, Wroclaw, Poland",
    "Dokerska 2a, Wroclaw, Poland",
    "Gwarna 6a, Wroclaw, Poland",
    "Horbaczewskiego 35, Wroclaw, Poland",
    "Ibn Siny Awicenny 53, Wroclaw, Poland",
    "Jaracza 75h, Wroclaw, Poland",
    "Kasprowicza 9a, Wroclaw, Poland",
    "Kiełczowska 70, Wroclaw, Poland",
    "Krzycka 94, Wroclaw, Poland",
    "Mińska 5, Wroclaw, Poland",
    "Młodych Techników 7, Wroclaw, Poland",
    "Olszewskiego 21, Wroclaw, Poland",
    "Opolska 131, Wroclaw, Poland",
    "Ostrowskiego 3, Wroclaw, Poland",
    "Eluarda 7, Wroclaw, Poland",
    "Pereca 20/1a, Wroclaw, Poland",
    "Piłsudskiego 4a, Wroclaw, Poland",
    "Popowicka 67, Wroclaw, Poland",
    "Powstańców Śląskich 168, Wroclaw, Poland",
    "Powstańców Śląskich 60, Wroclaw, Poland",
    "Rajska 71, Wroclaw, Poland",
    "Sienkiewicza 110, Wroclaw, Poland",
    "Strachocińska 159, Wroclaw, Poland",
    "Swojczycka 69, Wroclaw, Poland",
    "Traugutta 142, Wroclaw, Poland",
    "Trawowa 73, Wroclaw, Poland",
    "Warszawska 2, Wroclaw, Poland",
    "Weigla 12, Wroclaw, Poland",
    "Zakładowa 11h, Wroclaw, Poland",
    "Żelazna 34, Wroclaw, Poland",
    "Zwycięska 41, Wroclaw, Poland"
]

hospital_address = "Borowska 213, Wroclaw, Poland"

points = []

for addr in addresses:
    lat, lon = ox.geocode(addr)
    points.append({
        "name": addr,
        "lat": lat,
        "lon": lon,
        "type": "lab"
    })

lat, lon = ox.geocode(hospital_address)
points.append({
        "name": hospital_address,
        "lat": lat,
        "lon": lon,
        "type": "hospital"
    })



df = pd.DataFrame(points)
df.to_csv(OUT_DIR / "points_manual.csv", index=False, encoding="utf-8-sig")

# ============================================================
# 3. POBRANIE SIECI DRÓG WROCŁAWIA
#    network_type="drive" = drogi przejezdne dla samochodów
# ============================================================

print("Pobieram sieć drogową Wrocławia...")
G = ox.graph_from_place("Wrocław, Poland", network_type="drive")

# Zamiana grafu na GeoDataFrame
nodes_gdf, edges_gdf = ox.graph_to_gdfs(G)

print("Liczba węzłów:", len(nodes_gdf))
print("Liczba odcinków dróg:", len(edges_gdf))

# ============================================================
# 4. MAPA
# ============================================================

center_lat = df["lat"].mean()
center_lon = df["lon"].mean()

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=11,
    tiles="OpenStreetMap"
)

# ============================================================
# 5. RYSOWANIE DRÓG
# ============================================================

for _, edge in edges_gdf.iterrows():
    geom = edge.geometry
    if geom is None:
        continue

    coords = [(y, x) for x, y in geom.coords]

    folium.PolyLine(
        locations=coords,
        weight=1,
        opacity=0.25
    ).add_to(m)

# ============================================================
# 6. RYSOWANIE PUNKTÓW
# ============================================================

for idx, row in df.iterrows():
    if row["type"] == "hospital":
        folium.CircleMarker(
            location=[row["lat"], row["lon"]],
            radius=9,
            popup=f"<b>{row['name']}</b>",
            tooltip=row["name"],
            color="red",
            fill=True,
            fill_opacity=1.0
        ).add_to(m)
    else:
        folium.CircleMarker(
            location=[row["lat"], row["lon"]],
            radius=6,
            popup=f"{row['name']}",
            tooltip=row["name"],
            color="blue",
            fill=True,
            fill_opacity=0.9
        ).add_to(m)

        # numer obok punktu
        folium.Marker(
            [row["lat"], row["lon"]],
            icon=folium.DivIcon(
                html=f"""
                <div style="font-size: 10pt; color: black; font-weight: bold;">
                    {idx}
                </div>
                """
            )
        ).add_to(m)

# ============================================================
# 7. ZAPIS
# ============================================================

map_path = OUT_DIR / "wroclaw_roads_points.html"
m.save(map_path)

print("Gotowe.")
print("Mapa:", map_path.resolve())
print("Punkty:", (OUT_DIR / "points_manual.csv").resolve())

Pobieram sieć drogową Wrocławia...
Liczba węzłów: 7291
Liczba odcinków dróg: 17213
Gotowe.
Mapa: C:\Users\weron\Desktop\operations_reaserch\wroclaw_map_output\wroclaw_roads_points.html
Punkty: C:\Users\weron\Desktop\operations_reaserch\wroclaw_map_output\points_manual.csv


* pkty szpital i droga z jednego lab do szpitala

In [2]:
import networkx as nx
import osmnx as ox

# ============================================================
# 1. WYBIERAMY SZPITAL I JEDNO LABORATORIUM
# ============================================================

hospital = df[df["type"] == "hospital"].iloc[0]
lab = df[df["type"] == "lab"].iloc[0]  # pierwsze lab

print("Hospital:", hospital["name"])
print("Lab:", lab["name"])

# ============================================================
# 2. ZNAJDUJEMY NAJBLIŻSZE WĘZŁY W GRAFIE
# ============================================================

hospital_node = ox.distance.nearest_nodes(G, hospital["lon"], hospital["lat"])
lab_node = ox.distance.nearest_nodes(G, lab["lon"], lab["lat"])

# ============================================================
# 3. NAJKRÓTSZA TRASA (PO DROGACH!)
# ============================================================

route = nx.shortest_path(G, hospital_node, lab_node, weight="length")

# długość trasy
route_length = nx.shortest_path_length(G, hospital_node, lab_node, weight="length")

print(f"Długość trasy: {route_length/1000:.2f} km")

# ============================================================
# 4. KONWERSJA NA WSPÓŁRZĘDNE
# ============================================================

route_coords = [(G.nodes[node]["y"], G.nodes[node]["x"]) for node in route]

# ============================================================
# 5. MAPA Z TRASĄ
# ============================================================

m_route = folium.Map(
    location=[df["lat"].mean(), df["lon"].mean()],
    zoom_start=12
)

# punkty
for _, row in df.iterrows():
    color = "red" if row["type"] == "hospital" else "blue"
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=7,
        color=color,
        fill=True
    ).add_to(m_route)

# trasa
folium.PolyLine(
    locations=route_coords,
    weight=5,
    opacity=0.9,
    tooltip="Route hospital → lab"
).add_to(m_route)

# zapis
# route_map_path = OUT_DIR / "route_example.html"
# m_route.save(route_map_path)

# print(f"Mapa trasy zapisana: {route_map_path.resolve()}")

Hospital: Borowska 213, Wroclaw, Poland
Lab: pl. Grunwaldzki 18-20, Wroclaw, Poland
Długość trasy: 5.12 km


* generowanie danych

In [3]:
nodes = []
node_id = 0

for p in points:
    if p["type"] == "hospital":
        nodes.append({
            "node_id": 0,
            "name": "Central Hospital",
            "address": p["name"],
            "type": "hospital",
            "lat": p["lat"],
            "lon": p["lon"]
        })
    else:
        node_id += 1
        nodes.append({
            "node_id": node_id,
            "name": f"Lab {node_id}",
            "address": p["name"],
            "type": "lab",
            "lat": p["lat"],
            "lon": p["lon"]
        })

nodes_df = pd.DataFrame(nodes).sort_values("node_id").reset_index(drop=True)



requests = []

labs_df = nodes_df[nodes_df["type"] == "lab"].reset_index(drop=True)




###########################################################
######## TU ZMIENIAMY DANE O OGOLNIE O PROBKACH ###########
###########################################################

######## FIRST SET #########

for i, row in labs_df.iterrows():
    requests.append({
        "request_id": i + 1,
        "lab_node_id": int(row["node_id"]),
        "demand": 10,
        "ready_time": 0,          # można odbierać od początku
        "due_time": 180,          # szerokie okno
        "service_time": 5,
        "max_transport_time": 180 # szeroki limit
    })


######## SECOND SET #########

# for i, row in labs_df.iterrows():
#     ready_time = np.random.randint(0, 120)        # moment powstania próbki
#     due_time = ready_time + np.random.randint(60, 120)
#     requests.append({
#         "request_id": i + 1,
#         "lab_node_id": int(row["node_id"]),
#         "demand": 10,
#         "ready_time": ready_time,          # można odbierać od początku
#         "due_time": due_time,          # szerokie okno
#         "service_time": 5,
#         "max_transport_time": 180 # szeroki limit
#     })




requests_df = pd.DataFrame(requests)

############# A TU DOPASOWUJEMY LBE AUT ###############
######## FIRST SET OF CARS ######### ---> NOT WOKRING

vehicles = [
    {"vehicle_id": 1, "start_node": 0, "end_node": 0, "capacity": 9999, "shift_start": 0, "shift_end": 400},
    # {"vehicle_id": 2, "start_node": 0, "end_node": 0, "capacity": 9999, "shift_start": 0, "shift_end": 400},
    # {"vehicle_id": 3, "start_node": 0, "end_node": 0, "capacity": 9999, "shift_start": 0, "shift_end": 400},
]


######## SECOND SET OF CARS ######### ---> NOT WOKRING
# vehicles = [
#     {"vehicle_id": 1, "start_node": 0, "end_node": 0, "capacity": 9999, "shift_start": 0, "shift_end": 400},
#     {"vehicle_id": 2, "start_node": 0, "end_node": 0, "capacity": 9999, "shift_start": 0, "shift_end": 400},
#     {"vehicle_id": 3, "start_node": 0, "end_node": 0, "capacity": 9999, "shift_start": 0, "shift_end": 400},
#     {"vehicle_id": 4, "start_node": 0, "end_node": 0, "capacity": 9999, "shift_start": 0, "shift_end": 400},
#     {"vehicle_id": 5, "start_node": 0, "end_node": 0, "capacity": 9999, "shift_start": 0, "shift_end": 400},
#     {"vehicle_id": 6, "start_node": 0, "end_node": 0, "capacity": 9999, "shift_start": 0, "shift_end": 400},
#     {"vehicle_id": 7, "start_node": 0, "end_node": 0, "capacity": 9999, "shift_start": 0, "shift_end": 400},
#     {"vehicle_id": 8, "start_node": 0, "end_node": 0, "capacity": 9999, "shift_start": 0, "shift_end": 400},
#     {"vehicle_id": 9, "start_node": 0, "end_node": 0, "capacity": 9999, "shift_start": 0, "shift_end": 400},
# ]

vehicles_df = pd.DataFrame(vehicles)





# nodes_df.to_csv(f"{OUT_DIR}/nodes.csv", index=False, encoding="utf-8-sig")
# requests_df.to_csv(f"{OUT_DIR}/requests.csv", index=False, encoding="utf-8-sig")
# vehicles_df.to_csv(f"{OUT_DIR}/vehicles.csv", index=False, encoding="utf-8-sig")

* macierz czasu przejazdu miedzy wezlami 
(nastepny krok)

In [5]:
from pathlib import Path
import pandas as pd
import osmnx as ox
import networkx as nx
import numpy as np

# ============================================================
# 1. ŚCIEŻKI
# ============================================================

OUT_DIR.mkdir(exist_ok=True)

# nodes_path = nodes_df

# ============================================================
# 2. WCZYTANIE DANYCH
# ============================================================

# nodes_df = pd.read_csv(nodes_path)

print("Wczytane węzły:")
print(nodes_df[["node_id", "name", "type", "lat", "lon"]])

# ============================================================
# 3. POBRANIE GRAFU I DODANIE CZASÓW PRZEJAZDU
# ============================================================

print("Pobieram graf drogowy...")
G = ox.graph_from_place("Wrocław, Poland", network_type="drive")

# Dodanie prędkości i czasu przejazdu do krawędzi
G = ox.routing.add_edge_speeds(G)
G = ox.routing.add_edge_travel_times(G)

print("Graf gotowy.")

# ============================================================
# 4. PRZYPISANIE PUNKTÓW DO NAJBLIŻSZYCH WĘZŁÓW GRAFU
# ============================================================

# Uwaga: X = longitude, Y = latitude
nearest_nodes = ox.distance.nearest_nodes(
    G,
    X=nodes_df["lon"].values,
    Y=nodes_df["lat"].values
)

nodes_df["osmnx_node"] = nearest_nodes

nodes_df.to_csv(OUT_DIR / "nodes_with_osm_ids.csv", index=False, encoding="utf-8-sig")
print("Zapisano nodes_with_osm_ids.csv")

# ============================================================
# 5. MACIERZ CZASÓW PRZEJAZDU
# ============================================================

node_ids = nodes_df["node_id"].tolist()
names = nodes_df["name"].tolist()
osm_nodes = nodes_df["osmnx_node"].tolist()

n = len(nodes_df)

time_matrix = pd.DataFrame(
    data=np.full((n, n), np.nan),
    index=node_ids,
    columns=node_ids
)

distance_matrix = pd.DataFrame(
    data=np.full((n, n), np.nan),
    index=node_ids,
    columns=node_ids
)

print("Liczę macierz czasów przejazdu...")

for i in range(n):
    source_osm = osm_nodes[i]

    # najkrótsze czasy od jednego źródła do wszystkich
    travel_times = nx.single_source_dijkstra_path_length(
        G,
        source_osm,
        weight="travel_time"
    )

    distances = nx.single_source_dijkstra_path_length(
        G,
        source_osm,
        weight="length"
    )

    for j in range(n):
        target_osm = osm_nodes[j]

        if target_osm in travel_times:
            # travel_time w sekundach -> minuty
            time_matrix.iloc[i, j] = travel_times[target_osm] / 60.0

        if target_osm in distances:
            # length w metrach
            distance_matrix.iloc[i, j] = distances[target_osm]

# ============================================================
# 6. ZAPIS MACIERZY
# ============================================================

# time_matrix.to_csv(OUT_DIR / "travel_time_matrix_min.csv", encoding="utf-8-sig")
# distance_matrix.to_csv(OUT_DIR / "distance_matrix_m.csv", encoding="utf-8-sig")

print("Zapisano:")
# print(OUT_DIR / "travel_time_matrix_min.csv")
# print(OUT_DIR / "distance_matrix_m.csv")

# ============================================================
# 7. DODATKOWO: WERSJA 'LONG FORMAT'
# ============================================================

rows = []
for i in range(n):
    for j in range(n):
        rows.append({
            "from_node_id": node_ids[i],
            "from_name": names[i],
            "to_node_id": node_ids[j],
            "to_name": names[j],
            "travel_time_min": time_matrix.iloc[i, j],
            "distance_m": distance_matrix.iloc[i, j],
        })

travel_long_df = pd.DataFrame(rows)
travel_long_df.to_csv(OUT_DIR / "travel_times_long.csv", index=False, encoding="utf-8-sig")

print("Zapisano także:")
print(OUT_DIR / "travel_times_long.csv")
print("Gotowe.")

Wczytane węzły:
    node_id              name      type        lat        lon
0         0  Central Hospital  hospital  51.074509  17.031021
1         1             Lab 1       lab  51.111438  17.058332
2         2             Lab 2       lab  51.094712  17.014853
3         3             Lab 3       lab  51.120156  17.036893
4         4             Lab 4       lab  51.148379  17.118011
5         5             Lab 5       lab  51.136820  17.038254
6         6             Lab 6       lab  51.112600  16.959335
7         7             Lab 7       lab  51.056605  17.056557
8         8             Lab 8       lab  51.104307  17.115801
9         9             Lab 9       lab  51.138894  17.025801
10       10            Lab 10       lab  51.106972  17.046210
11       11            Lab 11       lab  51.137909  16.965286
12       12            Lab 12       lab  51.100954  17.038287
13       13            Lab 13       lab  51.124845  16.971391
14       14            Lab 14       lab  51.080271  16

# 2 prezentacja - modele

In [6]:
import pandas as pd
import numpy as np
from pulp import (
    LpProblem, LpMinimize, LpVariable, lpSum, LpBinary,
    LpStatus, value, PULP_CBC_CMD
)

# ============================================================
# 1. WCZYTANIE DANYCH
# ============================================================

# nodes_df = pd.read_csv(OUT_DIR /"nodes.csv")
# requests_df = pd.read_csv(OUT_DIR / "requests.csv")
# vehicles_df = pd.read_csv(OUT_DIR / "vehicles.csv")

# time_matrix = pd.read_csv(OUT_DIR / "travel_time_matrix_min.csv", index_col=0)
# distance_matrix = pd.read_csv(OUT_DIR / "distance_matrix_m.csv", index_col=0)

# indeksy/kolumny jako int
time_matrix.index = time_matrix.index.astype(int)
time_matrix.columns = time_matrix.columns.astype(int)

distance_matrix.index = distance_matrix.index.astype(int)
distance_matrix.columns = distance_matrix.columns.astype(int)

print("nodes:", nodes_df.shape)
print("requests:", requests_df.shape)
print("vehicles:", vehicles_df.shape)
print("time_matrix:", time_matrix.shape)

nodes: (41, 7)
requests: (40, 7)
vehicles: (1, 6)
time_matrix: (41, 41)


WYNIK

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path
from pulp import (
    LpProblem, LpMinimize, LpVariable, lpSum, LpBinary,
    LpStatus, value, PULP_CBC_CMD
)

# ============================================================
# 0. USTAWIENIA
# ============================================================

np.random.seed(42)


# ============================================================
# 1. DANE WEJŚCIOWE
# ============================================================
# Zakładam, że masz już w pamięci:
# - nodes_df
# - requests_df
# - vehicles_df
# - time_matrix
# - distance_matrix (opcjonalnie, tu nieużywana w funkcji celu)
#
# Upewniamy się, że indeksy macierzy czasu są int.

time_matrix = time_matrix.copy()
time_matrix.index = time_matrix.index.astype(int)
time_matrix.columns = time_matrix.columns.astype(int)

# ============================================================
# 2. INSTANCJA TESTOWA
# ============================================================

requests_small = requests_df.copy()
vehicles_small = vehicles_df.copy()

# Jeśli chcesz prostszy test, odkomentuj:
# requests_small["ready_time"] = 0
# requests_small["due_time"] = 300
# requests_small["service_time"] = 5
# requests_small["max_transport_time"] = 180

# ============================================================
# 3. WĘZŁY I ZBIORY
# ============================================================

hospital_id = 0
start_node = -1
end_node = -2

lab_nodes = requests_small["lab_node_id"].astype(int).tolist()
vehicles = vehicles_small["vehicle_id"].astype(int).tolist()

all_nodes = [start_node] + lab_nodes + [end_node]

# ============================================================
# 4. PARAMETRY
# ============================================================

ready_time = dict(zip(requests_small["lab_node_id"], requests_small["ready_time"]))
due_time = dict(zip(requests_small["lab_node_id"], requests_small["due_time"]))
service_time = dict(zip(requests_small["lab_node_id"], requests_small["service_time"]))
max_transport_time = dict(zip(requests_small["lab_node_id"], requests_small["max_transport_time"]))

shift_start = dict(zip(vehicles_small["vehicle_id"], vehicles_small["shift_start"]))
shift_end = dict(zip(vehicles_small["vehicle_id"], vehicles_small["shift_end"]))

# service time dla sztucznych węzłów
service_time[start_node] = 0
service_time[end_node] = 0

# duże M
M = 10000

# mała kara za użycie pojazdu
EPS_VEHICLE = 0.001

# ============================================================
# 5. MACIERZ CZASÓW PRZEJAZDU DLA SZTUCZNYCH WĘZŁÓW
# ============================================================
# start_node i end_node fizycznie odpowiadają szpitalowi (hospital_id = 0)

travel_cost = {}

for i in all_nodes:
    for j in all_nodes:
        if i == j:
            continue
        if j == start_node:
            continue   # do startu nie wjeżdżamy
        if i == end_node:
            continue   # z końca nie wyjeżdżamy

        real_i = hospital_id if i in [start_node, end_node] else i
        real_j = hospital_id if j in [start_node, end_node] else j

        travel_cost[(i, j)] = float(time_matrix.loc[real_i, real_j])

# pomocnicze listy łuków
arcs = [(i, j) for (i, j) in travel_cost.keys()]

# ============================================================
# 6. MODEL
# ============================================================

model = LpProblem("VRP_Biological_Samples_No_Spoke_StartEndSplit", LpMinimize)

# ------------------------------------------------------------
# Zmienne decyzyjne
# ------------------------------------------------------------

# x[k,i,j] = 1, jeśli pojazd k jedzie z i do j
x = LpVariable.dicts(
    "x",
    ((k, i, j) for k in vehicles for (i, j) in arcs),
    cat=LpBinary
)

# t[k,i] = czas rozpoczęcia obsługi węzła i przez pojazd k
t = LpVariable.dicts(
    "t",
    ((k, i) for k in vehicles for i in all_nodes),
    lowBound=0
)

# u[k,i] = 1, jeśli pojazd k obsługuje laboratorium i
u = LpVariable.dicts(
    "u",
    ((k, i) for k in vehicles for i in lab_nodes),
    cat=LpBinary
)

# y[k] = 1, jeśli pojazd k jest użyty
y = LpVariable.dicts(
    "y",
    (k for k in vehicles),
    cat=LpBinary
)

# ============================================================
# 7. FUNKCJA CELU
# ============================================================

model += (
    lpSum(
        travel_cost[(i, j)] * x[(k, i, j)]
        for k in vehicles
        for (i, j) in arcs
    )
    + EPS_VEHICLE * lpSum(y[k] for k in vehicles)
), "Minimize_Travel_Time_And_Vehicle_Use"

# ============================================================
# 8. KAŻDE LAB ODWIEDZONE DOKŁADNIE RAZ
# ============================================================

for i in lab_nodes:
    # dokładnie jedno wyjście z labu łącznie po wszystkich pojazdach
    model += lpSum(
        x[(k, i, j)]
        for k in vehicles
        for j in all_nodes
        if i != j and (i, j) in travel_cost
    ) == 1, f"leave_lab_once_{i}"

    # dokładnie jedno wejście do labu łącznie po wszystkich pojazdach
    model += lpSum(
        x[(k, j, i)]
        for k in vehicles
        for j in all_nodes
        if i != j and (j, i) in travel_cost
    ) == 1, f"enter_lab_once_{i}"

# ============================================================
# 9. START / KONIEC TRASY I UŻYCIE POJAZDU
# ============================================================

for k in vehicles:
    # jeśli pojazd użyty, to dokładnie raz wyjeżdża ze startu
    model += lpSum(
        x[(k, start_node, j)]
        for j in lab_nodes
        if (start_node, j) in travel_cost
    ) == y[k], f"start_equals_used_{k}"

    # jeśli pojazd użyty, to dokładnie raz wjeżdża do end_node
    model += lpSum(
        x[(k, i, end_node)]
        for i in lab_nodes
        if (i, end_node) in travel_cost
    ) == y[k], f"end_equals_used_{k}"

# ============================================================
# 10. FLOW CONSERVATION W LABORATORIACH
# ============================================================

for k in vehicles:
    for i in lab_nodes:
        model += (
            lpSum(
                x[(k, j, i)]
                for j in all_nodes
                if j != i and (j, i) in travel_cost
            )
            ==
            lpSum(
                x[(k, i, j)]
                for j in all_nodes
                if j != i and (i, j) in travel_cost
            )
        ), f"flow_{k}_{i}"

# ============================================================
# 11. ZAKAZY DLA START/END
# ============================================================

for k in vehicles:
    # do start_node nic nie wjeżdża
    model += lpSum(
        x[(k, i, start_node)]
        for i in all_nodes
        if i != start_node and (i, start_node) in travel_cost
    ) == 0, f"no_in_start_{k}"

    # z end_node nic nie wyjeżdża
    model += lpSum(
        x[(k, end_node, j)]
        for j in all_nodes
        if j != end_node and (end_node, j) in travel_cost
    ) == 0, f"no_out_end_{k}"

# ============================================================
# 12. POWIĄZANIE u Z ŁUKAMI
# ============================================================

for k in vehicles:
    for i in lab_nodes:
        model += u[(k, i)] == lpSum(
            x[(k, i, j)]
            for j in all_nodes
            if j != i and (i, j) in travel_cost
        ), f"u_link_{k}_{i}"

# ============================================================
# 13. OKNA CZASOWE
# ============================================================

for k in vehicles:
    for i in lab_nodes:
        model += t[(k, i)] >= ready_time[i] - M * (1 - u[(k, i)]), f"ready_{k}_{i}"
        model += t[(k, i)] <= due_time[i] + M * (1 - u[(k, i)]), f"due_{k}_{i}"

# ============================================================
# 14. CZAS STARTU I KOŃCA ZMIANY
# ============================================================

for k in vehicles:
    model += t[(k, start_node)] == shift_start[k], f"shift_start_{k}"
    model += t[(k, end_node)] <= shift_end[k], f"shift_end_{k}"

# ============================================================
# 15. PROPAGACJA CZASU
# ============================================================

for k in vehicles:
    for i in all_nodes:
        for j in all_nodes:
            if i == j:
                continue
            if (i, j) not in travel_cost:
                continue
            if j == start_node:
                continue
            if i == end_node:
                continue

            serv_i = service_time.get(i, 0)

            model += (
                t[(k, j)] >= t[(k, i)] + serv_i + travel_cost[(i, j)] - M * (1 - x[(k, i, j)])
            ), f"time_prop_{k}_{i}_{j}"

# ============================================================
# 16. OGRANICZENIE MAKS. CZASU TRANSPORTU PRÓBKI
# ============================================================
# Próbka z labu i po obsłudze ma trafić do HUB (end_node) w zadanym czasie.

for k in vehicles:
    for i in lab_nodes:
        model += (
            t[(k, end_node)] - (t[(k, i)] + service_time[i])
            <= max_transport_time[i] + M * (1 - u[(k, i)])
        ), f"max_transport_{k}_{i}"

# ============================================================
# 17. ROZWIĄZANIE
# ============================================================

solver = PULP_CBC_CMD(msg=True, timeLimit=300)
model.solve(solver)

print("=" * 60)
print("STATUS:", LpStatus[model.status])
print("OBJECTIVE:", value(model.objective))
print("=" * 60)

# ============================================================
# 18. RAPORT UŻYCIA POJAZDÓW
# ============================================================

used_vehicles = []
for k in vehicles:
    y_val = value(y[k])
    if y_val is None:
        y_val = 0
    print(f"Pojazd {k} użyty: {int(round(y_val))}")
    if y_val > 0.5:
        used_vehicles.append(k)

print("Liczba użytych pojazdów:", len(used_vehicles))
print("Użyte pojazdy:", used_vehicles)

# ============================================================
# 19. WYBRANE ŁUKI
# ============================================================

selected_arcs = []
for k in vehicles:
    for (i, j) in arcs:
        val = value(x[(k, i, j)])
        if val is not None and val > 0.5:
            selected_arcs.append({
                "vehicle_id": k,
                "from_node": i,
                "to_node": j,
                "travel_time_min": travel_cost[(i, j)]
            })

selected_arcs_df = pd.DataFrame(selected_arcs)

print("\nWybrane łuki:")
print(selected_arcs_df)

# ============================================================
# 20. ODTWORZENIE TRAS
# ============================================================

def build_vehicle_route(arcs_df, vehicle_id, start_node=-1, end_node=-2):
    sub = arcs_df[arcs_df["vehicle_id"] == vehicle_id].copy()
    if sub.empty:
        return []

    arc_map = dict(zip(sub["from_node"], sub["to_node"]))

    route = [start_node]
    current = start_node
    visited = set([start_node])

    while current in arc_map:
        nxt = arc_map[current]
        route.append(nxt)

        if nxt == end_node:
            break

        if nxt in visited:
            print(f"Uwaga: cykl dla pojazdu {vehicle_id}")
            break

        visited.add(nxt)
        current = nxt

    return route

routes = {}
for k in vehicles:
    routes[k] = build_vehicle_route(
        selected_arcs_df,
        vehicle_id=k,
        start_node=start_node,
        end_node=end_node
    )

print("\nTRASY:")
for k, r in routes.items():
    print(f"Pojazd {k}: {r}")

# ============================================================
# 21. CZASY ODWIEDZIN
# ============================================================

print("\nCZASY:")
for k in vehicles:
    if routes[k]:
        print(f"\nPojazd {k}:")
        for node in routes[k]:
            print(f"  node={node}, time={value(t[(k, node)])}")

# ============================================================
# 22. KONTROLA OBSŁUGI LABÓW
# ============================================================

lab_assignment_rows = []
for k in vehicles:
    for i in lab_nodes:
        u_val = value(u[(k, i)])
        if u_val is not None and u_val > 0.5:
            lab_assignment_rows.append({
                "vehicle_id": k,
                "lab_node_id": i,
                "visit_time": value(t[(k, i)]),
                "ready_time": ready_time[i],
                "due_time": due_time[i],
                "service_time": service_time[i],
                "max_transport_time": max_transport_time[i]
            })

lab_assignment_df = pd.DataFrame(lab_assignment_rows)

print("\nPrzypisanie labów do pojazdów:")
print(lab_assignment_df.sort_values(["vehicle_id", "visit_time"]))

# ============================================================
# 23. ZAPIS WYNIKÓW
# ============================================================

selected_arcs_df.to_csv(
    OUT_DIR / "milp_selected_arcs_no_spoke.csv",
    index=False,
    encoding="utf-8-sig"
)

routes_rows = []
for k, route in routes.items():
    for order, node in enumerate(route):
        routes_rows.append({
            "vehicle_id": k,
            "stop_order": order,
            "node": node
        })

routes_out_df = pd.DataFrame(routes_rows)
routes_out_df.to_csv(
    OUT_DIR / "milp_routes_no_spoke.csv",
    index=False,
    encoding="utf-8-sig"
)

lab_assignment_df.to_csv(
    OUT_DIR / "milp_lab_assignments_no_spoke.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nZapisano:")
print(OUT_DIR / "milp_selected_arcs_no_spoke.csv")
print(OUT_DIR / "milp_routes_no_spoke.csv")
print(OUT_DIR / "milp_lab_assignments_no_spoke.csv")

print("\nPodsumowanie:")
print("Objective:", value(model.objective))
if not selected_arcs_df.empty:
    print("Manual sum of travel_time_min:", selected_arcs_df["travel_time_min"].sum())
else:
    print("Manual sum of travel_time_min: brak łuków")


print("\nSPRAWDZENIE LIFETIME:")

lifetime_rows = []

for k in vehicles:
    for i in lab_nodes:
        if value(u[(k, i)]) > 0.5:
            pickup_time = value(t[(k, i)])
            delivery_time = value(t[(k, end_node)])

            transport_time = delivery_time - (pickup_time + service_time[i])
            max_time = max_transport_time[i]

            slack = max_time - transport_time

            lifetime_rows.append({
                "vehicle": k,
                "lab": i,
                "pickup_time": pickup_time,
                "delivery_time": delivery_time,
                "transport_time": transport_time,
                "max_allowed": max_time,
                "slack": slack
            })

lifetime_df = pd.DataFrame(lifetime_rows)

print(lifetime_df.sort_values("slack"))

STATUS: Not Solved
OBJECTIVE: 108.39606889302095
Pojazd 1 użyty: 0
Liczba użytych pojazdów: 0
Użyte pojazdy: []

Wybrane łuki:
    vehicle_id  from_node  to_node  travel_time_min
0            1          1       31         0.824865
1            1          2       29         1.209416
2            1          3       15         3.280957
3            1          4       17         2.506342
4            1          5       16         0.069191
5            1          6       23         3.184520
6            1          7       22         7.431533
7            1          8       21         2.024136
8            1          9        5         2.761084
9            1         10       12         1.784596
10           1         11       13         4.060669
11           1         12       34         1.828571
12           1         13       27         2.838093
13           1         14       35         3.258829
14           1         15       38         2.274110
15           1         16        9       

WIZUALIZACJA i zapis

In [8]:
import folium
import networkx as nx
import osmnx as ox
import pandas as pd

# ============================================================
# 1. PRZYGOTOWANIE MAPOWANIA NODE_ID -> WSPÓŁRZĘDNE / OSM NODE
# ============================================================

# jeśli nodes_df nie ma jeszcze osmnx_node, to go wyznacz
if "osmnx_node" not in nodes_df.columns:
    nearest_nodes = ox.distance.nearest_nodes(
        G,
        X=nodes_df["lon"].values,
        Y=nodes_df["lat"].values
    )
    nodes_df["osmnx_node"] = nearest_nodes

node_lookup = nodes_df.set_index("node_id").to_dict("index")

hospital_id = 0
start_node = -1
end_node = -2

# funkcja mapująca sztuczne węzły na fizyczny szpital
def real_node_id(node):
    if node in [start_node, end_node]:
        return hospital_id
    return node

# ============================================================
# 2. MAPA BAZOWA
# ============================================================

center_lat = nodes_df["lat"].mean()
center_lon = nodes_df["lon"].mean()

m_milp = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=11,
    tiles="OpenStreetMap"
)

# ============================================================
# 3. RYSOWANIE PUNKTÓW
# ============================================================

for _, row in nodes_df.iterrows():
    if row["type"] == "hospital":
        folium.CircleMarker(
            location=[row["lat"], row["lon"]],
            radius=9,
            popup=f"<b>{row['name']}</b>",
            tooltip="Hospital",
            color="red",
            fill=True,
            fill_opacity=1.0
        ).add_to(m_milp)
    else:
        folium.CircleMarker(
            location=[row["lat"], row["lon"]],
            radius=6,
            popup=f"{row['name']} (node_id={row['node_id']})",
            tooltip=f"{row['name']}",
            color="blue",
            fill=True,
            fill_opacity=0.9
        ).add_to(m_milp)

        folium.Marker(
            [row["lat"], row["lon"]],
            icon=folium.DivIcon(
                html=f"""
                <div style="font-size: 10pt; color: black; font-weight: bold;">
                    {int(row['node_id'])}
                </div>
                """
            )
        ).add_to(m_milp)

# ============================================================
# 4. KOLORY DLA POJAZDÓW
# ============================================================

vehicle_colors = {
    1: "green",
    2: "purple",
    3: "orange",
    4: "darkred",
    5: "cadetblue",
    6: "yellow"
}

# ============================================================
# 5. RYSOWANIE TRAS POJAZDÓW
# ============================================================

route_summary = []

for vehicle_id, route in routes.items():
    if not route or len(route) < 2:
        continue

    color = vehicle_colors.get(vehicle_id, "blue")

    total_length_m = 0
    total_time_min = 0

    # przejścia między kolejnymi punktami trasy
    for idx in range(len(route) - 1):
        from_model_node = route[idx]
        to_model_node = route[idx + 1]

        from_real = real_node_id(from_model_node)
        to_real = real_node_id(to_model_node)

        from_osm = node_lookup[from_real]["osmnx_node"]
        to_osm = node_lookup[to_real]["osmnx_node"]

        # najkrótsza trasa po drogach
        path = nx.shortest_path(G, from_osm, to_osm, weight="travel_time")

        # długość i czas odcinka
        seg_length_m = nx.shortest_path_length(G, from_osm, to_osm, weight="length")
        seg_time_sec = nx.shortest_path_length(G, from_osm, to_osm, weight="travel_time")

        total_length_m += seg_length_m
        total_time_min += seg_time_sec / 60.0

        # współrzędne odcinka
        path_coords = [(G.nodes[n]["y"], G.nodes[n]["x"]) for n in path]

        folium.PolyLine(
            locations=path_coords,
            weight=5,
            opacity=0.85,
            color=color,
            tooltip=f"Vehicle {vehicle_id}: {from_model_node} → {to_model_node}"
        ).add_to(m_milp)

    # zaznaczenie kolejności punktów na trasie
    for stop_order, model_node in enumerate(route):
        real_id = real_node_id(model_node)
        row = node_lookup[real_id]

        if model_node == start_node:
            label = f"V{vehicle_id}-START"
        elif model_node == end_node:
            label = f"V{vehicle_id}-END"
        else:
            label = f"V{vehicle_id}-{stop_order}"

        folium.Marker(
            [row["lat"], row["lon"]],
            icon=folium.DivIcon(
                html=f"""
                <div style="
                    font-size: 9pt;
                    color: {color};
                    font-weight: bold;
                    background-color: white;
                    border: 1px solid {color};
                    border-radius: 4px;
                    padding: 1px 3px;
                ">
                    {label}
                </div>
                """
            )
        ).add_to(m_milp)

    route_summary.append({
        "vehicle_id": vehicle_id,
        "route": " -> ".join(map(str, route)),
        "distance_km": round(total_length_m / 1000.0, 2),
        "travel_time_min": round(total_time_min, 2)
    })

# ============================================================
# 6. PODSUMOWANIE
# ============================================================

route_summary_df = pd.DataFrame(route_summary)

print("Podsumowanie tras:")
print(route_summary_df)

# route_summary_df.to_csv(
#     OUT_DIR / "milp_route_summary_no_spoke.csv",
#     index=False,
#     encoding="utf-8-sig"
# )

# ============================================================
# 7. ZAPIS MAPY
# ============================================================

map_path = OUT_DIR / "milp_routes_no_spoke_map.html"
m_milp.save(map_path)

print("\nZapisano mapę:")
print(map_path.resolve())


Podsumowanie tras:
Empty DataFrame
Columns: []
Index: []

Zapisano mapę:
C:\Users\weron\Desktop\operations_reaserch\wroclaw_map_output\milp_routes_no_spoke_map.html


WHAT IF:

* more cars
* more requests